# ModernBERT-JEPA decider: JEPA pre-training with LoRA

Run 9 (`03_jepa_pretrain.ipynb`) found that pre-training all of ModernBERT on our states, with JEPA or MLM, made the decision model worse than plain ModernBERT-base (JevBench 0.307 / 0.329 against 0.394). This run keeps ModernBERT's own weights frozen and lets JEPA train only **LoRA adapters** (rank 64 on the attention and MLP projections), so the encoder cannot drift far from what it already knows.

**Stage 0.** The context encoder (ModernBERT + adapters) reads a state with spans hidden; the predictor predicts the target's vectors at the hidden positions; the target is the same frozen ModernBERT with an **EMA copy of the adapters**, reading the full state. The target is harder than in run 9: spans of 3–8 words, about 25% of the words, mostly starting at numbers. Logged: the cosine, the target's spread, and a **retrieval check** (does a prediction match its own hidden token better than the other hidden tokens in the batch?).

**Merge.** The adapters are merged into ModernBERT at full strength (1.0) and at half strength (0.5).

**Stage 1.** Run 8's arm B decision training from each merged encoder, scored on JevBench, our dev and Typed Decisions' train split; compared with plain ModernBERT (run 8 B) and full-update JEPA (run 9).

## Parameters

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

ENCODER = env("ENCODER", "answerdotai/ModernBERT-base")
DATASET = "skamalj/signpost-d-decisions"           # our generated data (private): train and dev
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"   # pinned public items
TASKSOURCE = "tasksource/tasksource-jev-typed-decisions"       # Stage 1: public typed decisions
TS_QUESTIONS = int(env("TS_QUESTIONS", "50000"))   # tasksource sample size
TS_PER_SOURCE = int(env("TS_PER_SOURCE", "400"))   # at most this many from one source
TS_BUFFER = 50000                                  # streaming shuffle buffer
OVERLAP_N = 8                                      # drop a state sharing an 8-word run with a test
TRAIN_QUESTIONS = int(env("TRAIN_QUESTIONS", "0"))  # our questions to train on; 0 = all 12,274
MAX_OPTIONS = 26                                   # letters A-Z
BATCH = int(env("BATCH", "16"))                    # questions per step
TRAIN_MAX_LEN = 1024
EVAL_MAX_LEN = 8192                                # JevBench's hard items are long documents
OPTION_MAX_TOKENS = 64                             # each option's text is cut to this
EVAL_TOKEN_BUDGET = 16384                          # tokens per evaluation batch
LR_ENCODER = float(env("LR_ENCODER", "5e-5"))      # top layer; lower layers decay by LLRD
LLRD = float(env("LLRD", "0.95"))                  # gentle, so the whole encoder trains
ARM_EPOCHS = int(env("ARM_EPOCHS", "2"))           # passes over each arm's training data
PRE_MAX_LEN = 512                                  # Stage 0 state length
PRE_EPOCHS = int(env("PRE_EPOCHS", "1"))           # passes over the states in Stage 0
PRE_BATCH = 32
PRE_LR = 5e-5                                      # Stage 0 encoder rate (predictor 5e-4)
MASK_RATIO, SPAN_MIN, SPAN_MAX = 0.25, 3, 8       # share of words masked, span length
NUMERIC_START = 0.7                                # share of spans that start at a number
LORA_RANK = 64
MERGE_SCALES = [1.0, 0.5]                          # strengths at which the adapters are merged
CURVE_POINTS = 4                                   # scored at each quarter of an arm's training
SANITY_TRAIN, SANITY_TEST = 300, 200               # the sanity test: the answer is in the state
OVERFIT_N, OVERFIT_EPOCHS, OVERFIT_EVERY = 200, 30, 5   # the fair overfit check
OVERFIT_LR = 5e-5                                  # constant, after a short warm-up
LR_HEAD = float(env("LR_HEAD", "5e-4"))            # predictor, temperatures, biases
WARMUP = 0.05
EMA_START, EMA_END = 0.996, 0.9999                 # target-encoder momentum, cosine schedule
LAMBDA_JEPA, LAMBDA_ORD = 0.5, 0.5
LAMBDA_VAR = float(env("LAMBDA_VAR", "0"))         # variance regulariser on ẑ (ablation)
LAMBDA_COV = float(env("LAMBDA_COV", "0"))         # covariance term (ablation; noisy at batch 16)
SEED = int(env("SEED", "42"))

## Packages and the Hugging Face token

The token reads our private dataset: from the environment, a Kaggle or Colab secret, or a hidden prompt. Only where it came from is printed.

In [ ]:
!pip install -q "transformers>=4.48" peft

import copy, gc, json, math, random, re, statistics, time, urllib.request
from collections import Counter, defaultdict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret"
    except Exception:
        pass
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token, "Colab secret"
    except Exception:
        pass
    import getpass
    token = getpass.getpass("Hugging Face token (read access): ").strip()
    return (token, "the prompt") if token else (None, "NOT FOUND")


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {token_from}")

## Device and log

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 where the GPU has it; fp32 otherwise (ModernBERT overflows in fp16)
BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


gpu = torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU"
log(f"torch {torch.__version__} | transformers {transformers.__version__} | {gpu} | "
    f"{'bf16' if BF16 else 'fp32'} | {ENCODER}")

## One item per question

Every question becomes: question text, state text, option texts and a gold distribution over the options. `noul` options are the criteria's `true` / `false` descriptions (or nothing), shown as `yes` / `no`; `choice` options are `name - description`, shown as `A`, `B`, …; `score` levels are shown as `0`, `1`, …

In [ ]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def option_texts(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        return ["true", "false"], [crit.get("true") or "", crit.get("false") or ""]
    if kind == "choice":
        names = list(crit)
        return names, [f"{n} - {d}" if d else n for n, d in crit.items()]
    return [str(i) for i in range(len(crit))], [str(level) for level in crit]


def make_item(kind, question, state, q, gold, source, family=""):
    names, texts = option_texts(q)
    total = sum(gold)
    return {"type": kind, "question": question or "", "state": state_text(state), "names": names,
            "texts": texts, "gold": [g / total for g in gold], "source": source, "family": family}

## Our training and dev data

In [ ]:
def load_split(split):
    path = hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN)
    items = []
    for line in open(path, encoding="utf-8"):
        rec = json.loads(line)
        for qid, q in rec["questions"].items():
            probs = rec["gold"][qid]["probabilities"]
            names, _ = option_texts(q)
            if len(names) > MAX_OPTIONS:
                continue
            gold = [float(probs[n]) for n in names]
            items.append(make_item(q["type"], q.get("instructions"), rec["state"], q, gold, split))
    return items


train_all, dev_items = load_split("train"), load_split("dev")
train_items = (random.Random(SEED).sample(train_all, min(TRAIN_QUESTIONS, len(train_all)))
               if TRAIN_QUESTIONS else train_all)
print(f"train {len(train_items)} of {len(train_all)} questions "
      f"{dict(Counter(it['type'] for it in train_items))} | dev {len(dev_items)}")

## JevBench public items (test only, never trained on)

One question per item and one expected label, so the gold is one-hot.

In [ ]:
def load_jevbench(split):
    url = (f"https://raw.githubusercontent.com/fstandhartinger/jevbench/{JEVBENCH_COMMIT}"
           f"/datasets/public/{split}.jsonl")
    items = []
    for line in urllib.request.urlopen(url).read().decode("utf-8").splitlines():
        rec = json.loads(line)
        q = rec["question"]
        names, _ = option_texts(q)
        if q["type"] == "noul":
            target = "true" if rec["expected"] == "yes" else "false"
        else:
            target = str(rec["expected"])
        gold = [1.0 if n == target else 0.0 for n in names]
        assert sum(gold) == 1, rec["id"]
        items.append(make_item(q["type"], q["instructions"], rec["state"], q, gold,
                               split, rec["family"]))
    return items


jev_items = [it for split in ("original", "easy", "hard") for it in load_jevbench(split)]
print(f"JevBench public: {len(jev_items)} items {dict(Counter(it['source'] for it in jev_items))} "
      f"{dict(Counter(it['type'] for it in jev_items))}")

## Stage 1 data: public typed decisions, filtered

A guard set of word 8-grams from every test state we care about (JevBench's public items, Typed Decisions' train and test splits). Then the tasksource stream, shuffled: rows whose `license_use` is commercial, with 2–26 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set.

In [ ]:
WORD = re.compile(r"\w+")


def ngrams(text, n=OVERLAP_N):
    words = WORD.findall(text.lower())
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


guard = set()
for it in jev_items:
    guard |= ngrams(it["state"])
for split in ("train", "test"):
    path = hf_hub_download("LocalLLaMA/typed-decisions", f"all/{split}-00000-of-00001.parquet",
                           repo_type="dataset")
    for state in pd.read_parquet(path)["state"]:
        guard |= ngrams(state_text(json.loads(state)))
print(f"guard set: {len(guard):,} 8-grams")

In [ ]:
def ts_item(row):
    """A tasksource row -> one item, or None if it does not fit."""
    kind, opts, target = row["kind"], row["options"], row["target"]
    if kind == "noul":
        q, gold = {"type": "noul", "criteria": None}, [target[0], 1 - target[0]]
    elif kind == "choice" and 2 <= len(opts) <= MAX_OPTIONS and len(set(opts)) == len(opts):
        q, gold = {"type": "choice", "criteria": {o: None for o in opts}}, list(target)
    elif kind == "score" and 2 <= len(opts) <= 10:
        q, gold = {"type": "score", "criteria": list(opts)}, list(target)
    else:
        return None
    if sum(gold) <= 0 or len(gold) != len(option_texts(q)[0]):
        return None
    return make_item(kind, row["question"], row["state"], q, gold, "tasksource", row["source"])


ts_items, per_source, dropped = [], Counter(), Counter()
stream = load_dataset(TASKSOURCE, "default", split="train", streaming=True, token=HF_TOKEN)
stream = stream.shuffle(seed=SEED, buffer_size=TS_BUFFER)
for row in stream:
    if row["license_use"] != "commercial":
        dropped["licence"] += 1
        continue
    if per_source[row["source"]] >= TS_PER_SOURCE:
        continue
    item = ts_item(row)
    if item is None:
        dropped["shape"] += 1
        continue
    if ngrams(item["state"]) & guard:
        dropped["overlap"] += 1
        continue
    ts_items.append(item)
    per_source[row["source"]] += 1
    if len(ts_items) >= TS_QUESTIONS:
        break
print(f"Stage 1: {len(ts_items)} questions from {len(per_source)} sources "
      f"{dict(Counter(it['type'] for it in ts_items))} | dropped {dict(dropped)}")

## Label tokens

The 38 labels are A–Z, yes / no and 0–9. Each must be one token of ModernBERT's vocabulary (checked here, with the leading space it has after `Answer:`): the head starts from those tokens' pretrained output rows, and the gold label's token fills the slot in C2's target input.

In [ ]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD, MASK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id
LABELS = {"choice": [chr(65 + i) for i in range(MAX_OPTIONS)],
          "noul": ["yes", "no"],
          "score": [str(i) for i in range(10)]}


def one_token(label):
    ids = tok(" " + label, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"label {label!r} is {len(ids)} tokens"
    return ids[0]


LABEL_IDS = {kind: [one_token(lab) for lab in labs] for kind, labs in LABELS.items()}
HEAD_TOKENS = LABEL_IDS["choice"] + LABEL_IDS["noul"] + LABEL_IDS["score"]   # the 38 head outputs
HEAD_SLOTS = {"choice": list(range(26)), "noul": [26, 27], "score": list(range(28, 38))}
print({k: v[:4] for k, v in LABEL_IDS.items()}, "| head outputs", len(HEAD_TOKENS))

## One sequence per question

`[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`. The question and the options are kept whole (each option cut at `OPTION_MAX_TOKENS`); if the sequence is too long, the state's end is cut, and the cut is counted. With `gold_fill`, the gold label's token stands in place of `[MASK]` (the JEPA target's input). In training, choice options are shuffled, so a letter never carries a fixed answer.

In [ ]:
def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def shuffled(item, rng):
    """A copy of a choice item with its options in a random order (gold moved with them)."""
    if item["type"] != "choice" or rng is None:
        return item
    order = list(range(len(item["names"])))
    rng.shuffle(order)
    return {**item, "names": [item["names"][i] for i in order],
            "texts": [item["texts"][i] for i in order], "gold": [item["gold"][i] for i in order]}


def sequence(item, max_len):
    """-> (token ids with [MASK], mask position, gold label token, state was cut)."""
    labels = LABELS[item["type"]]
    tail = []
    for label, text in zip(labels, item["texts"]):
        line = f"{label}: {text}".strip() if text else label
        tail += ids(line)[:OPTION_MAX_TOKENS] + [SEP]
    tail += ids("Answer:")
    q = ids(item["question"])[:256]
    s = ids(item["state"])
    room = max_len - len(q) - len(tail) - 5
    cut = len(s) > room
    s = s[:max(room, 0)]
    seq = [CLS] + q + [SEP] + s + [SEP] + tail
    pos = len(seq)
    seq += [MASK, SEP]
    best = max(range(len(item["gold"])), key=item["gold"].__getitem__)
    gold_label = LABEL_IDS[item["type"]][best]
    return seq, pos, gold_label, cut


def batch_tensors(items, max_len, gold_fill=False):
    rows = [sequence(it, max_len) for it in items]
    width = max(len(r[0]) for r in rows)
    input_ids, att, pos = [], [], []
    for seq, p, gold_label, _ in rows:
        if gold_fill:
            seq = seq[:p] + [gold_label] + seq[p + 1:]
        input_ids.append(seq + [PAD] * (width - len(seq)))
        att.append([1] * len(seq) + [0] * (width - len(seq)))
        pos.append(p)
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(pos), sum(r[3] for r in rows)

## The model

- **Encoder** (trained): ModernBERT-base; its pretrained MLM transform turns the `[MASK]` vector into the space of token outputs.
- **Label head** (trained): 38 outputs (A–Z, yes / no, 0–9), started from the pretrained output rows of those tokens; each question keeps only its own labels.
- **Predictor** (trained): an MLP with a residual from the `[MASK]` vector to ẑ.
- **Target encoder** (EMA copy of the encoder, no gradients, always in eval mode): reads the gold-filled sequence; its vector at the answer position is the JEPA target. It is used only in training.

In [ ]:
TYPES = ["noul", "choice", "score"]


class MaskedDecider(nn.Module):
    def __init__(self, encoder_state=None):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(ENCODER, attn_implementation="sdpa")
        if encoder_state is not None:                 # an encoder from Stage 0
            self.mlm.model.load_state_dict(encoder_state)
        hidden = self.mlm.config.hidden_size
        for p in self.mlm.model.embeddings.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.label_head = nn.Linear(hidden, len(HEAD_TOKENS))
        with torch.no_grad():                         # start from the pretrained output rows
            self.label_head.weight.copy_(self.mlm.decoder.weight[HEAD_TOKENS])
            if self.mlm.decoder.bias is not None:
                self.label_head.bias.copy_(self.mlm.decoder.bias[HEAD_TOKENS])
        self.target = copy.deepcopy(self.mlm.model).requires_grad_(False)

    def mask_vectors(self, input_ids, att, pos):
        h = self.mlm.model(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

    def label_logits(self, vectors, items):
        """The 38-way head at [MASK]; each question keeps its own labels -> [items, options]."""
        all38 = self.label_head(self.mlm.head(vectors)).float()
        width = max(len(it["names"]) for it in items)
        out = torch.full((len(items), width), -1e4, device=vectors.device)
        for i, it in enumerate(items):
            k = len(it["names"])
            out[i, :k] = all38[i, HEAD_SLOTS[it["type"]][:k]]
        return out

    def train(self, mode=True):
        """Train the online parts; the EMA target encoder always stays in eval mode."""
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def target_vectors(self, input_ids, att, pos):
        h = self.target(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

## Scores and losses for a batch

KL to the gold; the ordinal error for `score`; with JEPA on, 1 − cos between the predicted vector and the target's vector at the answer position. Variance and covariance regularisers are off by default (ablations).

In [ ]:
def forward(model, items, max_len, jepa):
    input_ids, att, pos, cut = batch_tensors(items, max_len)
    v = model.mask_vectors(input_ids, att, pos)
    scores = model.label_logits(v, items)
    extra = {}
    if jepa:
        t_ids, t_att, t_pos, _ = batch_tensors(items, max_len, gold_fill=True)
        extra["target"] = model.target_vectors(t_ids, t_att, t_pos).float()
        extra["z"] = (v + model.predictor(v)).float()
    return scores, extra, cut


def gold_matrix(items, width):
    g = torch.zeros(len(items), width, device=DEVICE)
    for i, it in enumerate(items):
        g[i, :len(it["gold"])] = torch.tensor(it["gold"], device=DEVICE)
    return g


def losses(scores, extra, items):
    g = gold_matrix(items, scores.shape[1])
    logp = F.log_softmax(scores, dim=1)
    dec = (g * (g.clamp(min=1e-12).log() - logp)).sum(1).mean()
    parts = {"dec": dec}
    is_score = torch.tensor([it["type"] == "score" for it in items], device=DEVICE)
    if is_score.any():
        levels = torch.arange(scores.shape[1], device=DEVICE, dtype=torch.float32)
        n = torch.tensor([len(it["gold"]) - 1 for it in items], device=DEVICE).clamp(min=1)
        gap = ((logp.exp() - g) * levels).sum(1).abs() / n
        parts["ord"] = gap[is_score].mean()
    if "z" in extra:
        z = extra["z"]
        parts["jepa"] = (1 - F.cosine_similarity(z, extra["target"], dim=-1)).mean()
        zc = z - z.mean(0)
        if LAMBDA_VAR:
            parts["var"] = F.relu(1 - (zc.var(0) + 1e-4).sqrt()).mean()
        if LAMBDA_COV:
            cov = (zc.T @ zc) / max(len(items) - 1, 1)
            parts["cov"] = (cov - torch.diag(torch.diag(cov))).pow(2).sum() / z.shape[1]
    weights = {"dec": 1.0, "ord": LAMBDA_ORD, "jepa": LAMBDA_JEPA, "var": LAMBDA_VAR,
               "cov": LAMBDA_COV}
    total = sum(weights[k] * v for k, v in parts.items())
    return total, {k: v.item() for k, v in parts.items()}

## The measures

JevBench's: accuracy (argmax), multi-class Brier, top-label ECE over 10 equal-width bins (empty bins left out), ordinal MAE for `score`; and KL for our soft dev gold.

In [ ]:
def measure(rows):
    n = len(rows)
    bins = defaultdict(list)
    for r in rows:
        bins[min(int(r["conf"] * 10), 9)].append(r)
    ece = 0.0
    for b in bins.values():
        right = sum(x["ok"] for x in b) / len(b)
        sure = sum(x["conf"] for x in b) / len(b)
        ece += len(b) / n * abs(right - sure)
    out = {"n": n, "acc": round(sum(r["ok"] for r in rows) / n, 4),
           "brier": round(statistics.mean(r["brier"] for r in rows), 4),
           "ece": round(ece, 4), "kl": round(statistics.mean(r["kl"] for r in rows), 4)}
    mae = [r["mae"] for r in rows if r["mae"] is not None]
    if mae:
        out["ordinal_mae"] = round(statistics.mean(mae), 3)
    return out


def row_of(it, p):
    g = it["gold"]
    top, want = max(range(len(p)), key=p.__getitem__), max(range(len(g)), key=g.__getitem__)
    mae = abs(sum(i * x for i, x in enumerate(p)) - want) if it["type"] == "score" else None
    return {"ok": top == want, "conf": p[top], "margin": None,
            "brier": sum((a - b) ** 2 for a, b in zip(p, g)),
            "kl": sum(b * math.log(b / max(a, 1e-9)) for a, b in zip(p, g) if b > 0),
            "mae": mae, "type": it["type"], "source": it["source"], "family": it["family"]}

## Evaluation

Batches by token budget (long items alone), options in their given order, the measures overall and by split, type and family, and the time per item.

In [ ]:
@torch.no_grad()
def evaluate(model, items, max_len, name):
    model.eval()
    lengths = [len(sequence(it, max_len)[0]) for it in items]
    order = sorted(range(len(items)), key=lengths.__getitem__)
    rows, cut, start, t0 = [None] * len(items), 0, 0, time.perf_counter()
    while start < len(order):
        batch = [order[start]]
        while (start + len(batch) < len(order)
               and lengths[order[start + len(batch)]] * (len(batch) + 1) <= EVAL_TOKEN_BUDGET):
            batch.append(order[start + len(batch)])
        group = [items[i] for i in batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, _, c = forward(model, group, max_len, jepa=False)
        cut += c
        for i, it, p in zip(batch, group, F.softmax(scores, dim=1).tolist()):
            rows[i] = row_of(it, p[:len(it["gold"])])
        start += len(batch)
    ms = (time.perf_counter() - t0) / len(items) * 1000
    result = {"all": measure(rows), "ms_per_item": round(ms, 1), "states_cut": cut}
    for key in ("source", "type", "family"):
        values = sorted({r[key] for r in rows})
        result[key] = {v: measure([r for r in rows if r[key] == v]) for v in values}
    log(f"{name}: {result['all']} | {result['ms_per_item']} ms/item | states cut {cut}")
    for key in ("source", "type"):
        log("   " + " | ".join(f"{k} {v['acc']}" for k, v in result[key].items()))
    return result

## Optimizer, schedule and EMA

Layer-wise learning-rate decay from the top encoder layer; the MLM transform at the encoder rate; the label head and predictor at a higher rate; warm-up then cosine decay across both stages. After every step the target encoder moves towards the encoder.

In [ ]:
def make_optimizer(model, total_steps, lr=None, constant=False):
    lr = lr or LR_ENCODER
    layers = model.mlm.model.layers
    groups = [{"params": [p for p in layer.parameters() if p.requires_grad],
               "lr": lr * LLRD ** (len(layers) - 1 - i)} for i, layer in enumerate(layers)]
    in_layers = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in model.mlm.parameters() if p.requires_grad and id(p) not in in_layers]
    groups.append({"params": rest, "lr": lr})
    extra = list(model.predictor.parameters()) + list(model.label_head.parameters())
    groups.append({"params": extra, "lr": LR_HEAD})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    warm = max(1, int(WARMUP * total_steps))

    def factor(s):
        if s < warm:
            return (s + 1) / warm
        if constant:
            return 1.0
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, factor)


@torch.no_grad()
def ema_update(model, step, total_steps):
    m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / total_steps) + 1) / 2
    for pt, ps in zip(model.target.parameters(), model.mlm.model.parameters()):
        pt.mul_(m).add_(ps.detach(), alpha=1 - m)
    return m

## Training

A plan is a list of batches (`epochs` passes, reshuffled each pass); training runs a plan, or a slice of one, with choice options shuffled per item. Every 50 steps: the decision loss averaged over the last 50 batches, the current batch's losses, the top learning rate and, with JEPA on, cos(prediction, target), the spread and norm of both, and the EMA momentum.

In [ ]:
def make_plan(n, epochs, seed):
    """Batches of item indices: `epochs` passes over n items, reshuffled each pass."""
    rng, plan = random.Random(seed), []
    for _ in range(epochs):
        order = list(range(n))
        rng.shuffle(order)
        plan += [order[b:b + BATCH] for b in range(0, n, BATCH)]
    return plan


def train_stage(model, opt, sched, state, items, name, jepa, epochs=1, shuffle_options=True,
                quiet=False, plan=None):
    model.train()
    recent = []
    rng = random.Random(SEED + len(items) + state["step"])
    t0 = time.perf_counter()
    batches = plan if plan is not None else make_plan(len(items), epochs, SEED + len(items))
    for chunk in batches:
        batch = [shuffled(items[i], rng if shuffle_options else None) for i in chunk]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, extra, _ = forward(model, batch, TRAIN_MAX_LEN, jepa)
        loss, parts = losses(scores, extra, batch)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        params = [p for g in opt.param_groups for p in g["params"]]
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        sched.step()
        state["step"] += 1
        m = ema_update(model, state["step"], state["total"]) if jepa else 0.0
        recent = (recent + [parts["dec"]])[-50:]
        if state["step"] % 50 == 0 and not quiet:
            terms = " ".join(f"{k} {v:.4f}" for k, v in parts.items())
            more = ""
            if jepa:
                z, t = extra["z"], extra["target"]
                cos = F.cosine_similarity(z, t, dim=-1).mean().item()
                more = (f" | cos {cos:.3f}"
                        f" | z std {z.std(0).mean().item():.3f}"
                        f" norm {z.norm(dim=-1).mean().item():.1f}"
                        f" | t std {t.std(0).mean().item():.3f}"
                        f" norm {t.norm(dim=-1).mean().item():.1f} | ema {m:.5f}")
            top_lr = max(g["lr"] for g in opt.param_groups[:-1])
            log(f"{name} step {state['step']}/{state['total']} | decision loss, last 50 "
                f"{statistics.mean(recent):.4f} | batch: {terms} | grad {norm:.2f} | "
                f"lr {top_lr:.1e}{more}")
    if not quiet:
        log(f"{name} done in {(time.perf_counter() - t0) / 60:.1f} min")

## Checks before training

On a small batch from every question type: the target encoder gets no gradients and stays in eval mode after `train()`; its outputs are deterministic; the EMA step moves it towards the encoder; the gold label never appears in the context input; every loss is finite. Then the parameter counts: the inference model (encoder, MLM transform, label head) apart from the training-only predictor and EMA copy.

In [ ]:
def checks():
    model = MaskedDecider().to(DEVICE)
    sample = [next(it for it in train_items if it["type"] == k) for k in TYPES]
    sample += [next(it for it in ts_items if it["type"] == k) for k in TYPES] if ts_items else []
    model.train()
    assert not model.target.training, "target encoder left in training mode"
    assert not any(p.requires_grad for p in model.target.parameters())
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
        scores, extra, _ = forward(model, sample, TRAIN_MAX_LEN, jepa=True)
        again = forward(model, sample, TRAIN_MAX_LEN, jepa=True)[1]["target"]
    assert torch.allclose(extra["target"], again), "target outputs are not deterministic"
    loss, parts = losses(scores, extra, sample)
    assert all(math.isfinite(v) for v in parts.values()), parts
    loss.backward()
    assert all(p.grad is None for p in model.target.parameters()), "target received gradients"
    for it in sample:
        seq, pos, gold_label, _ = sequence(it, TRAIN_MAX_LEN)
        assert seq[pos] == MASK and gold_label not in seq[pos:], "gold label in the context"
    pairs = list(zip(model.target.parameters(), model.mlm.model.parameters()))
    p_t, p_o = next((t, o) for t, o in pairs if o.requires_grad)
    with torch.no_grad():
        p_o.add_(0.01)
    before = (p_t - p_o).abs().sum().item()
    ema_update(model, 1, 10)
    after = (p_t - p_o).abs().sum().item()
    assert after < before, "EMA did not move the target towards the encoder"
    count = lambda ps: sum(p.numel() for p in ps) / 1e6
    inference = count(model.mlm.parameters()) + count(model.label_head.parameters())
    log(f"checks passed | parts {parts} | inference model {inference:.1f} M | "
        f"predictor {count(model.predictor.parameters()):.1f} M | EMA copy (training only) "
        f"{count(model.target.parameters()):.1f} M")
    del model
    gc.collect()
    torch.cuda.empty_cache()


checks()

## One arm, with a learning curve

From the pretrained weights: one learning-rate schedule over `ARM_EPOCHS` passes of the arm's items; at each quarter, 1,000 of its own training questions, our dev and JevBench.

In [ ]:
def point(model, name, label, seen):
    """(label, training questions, dev, JevBench) at one point of the curve."""
    return (label,
            evaluate(model, seen, TRAIN_MAX_LEN, f"{name} train, {label}"),
            evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev, {label}"),
            evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench, {label}"))


def run(name, items, jepa=False, encoder_state=None):
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider(encoder_state).to(DEVICE)
    seen = random.Random(SEED).sample(items, 1000)
    plan = make_plan(len(items), ARM_EPOCHS, SEED + len(items))
    opt, sched = make_optimizer(model, len(plan))
    state = {"step": 0, "total": len(plan)}
    curve = [point(model, name, "untrained", seen)]
    cuts = [round(len(plan) * k / CURVE_POINTS) for k in range(CURVE_POINTS + 1)]
    for k in range(CURVE_POINTS):
        train_stage(model, opt, sched, state, items, name, jepa, plan=plan[cuts[k]:cuts[k + 1]])
        curve.append(point(model, name, f"{(k + 1) * ARM_EPOCHS / CURVE_POINTS:g} passes", seen))
    return model, curve

## Stage 0 data: states only

The unique states of the questions loaded above (tasksource and our records), at most `PRE_MAX_LEN` tokens; no questions, answers or benchmark states.

In [ ]:
seen_states, pre_states = set(), []
for it in ts_items + train_items:
    if it["state"] not in seen_states:
        seen_states.add(it["state"])
        pre_states.append(it["state"])
random.Random(SEED).shuffle(pre_states)
print(f"Stage 0: {len(pre_states)} unique states")

## Masking spans of a state

About `MASK_RATIO` of the words are hidden, in spans of `SPAN_MIN`–`SPAN_MAX` words; a share `NUMERIC_START` of the spans start at a word containing a digit when there is one. Returns the token ids and which tokens are hidden.

In [ ]:
DIGIT = re.compile(r"\d")


def masked_state(text, rng):
    enc = tok(text, add_special_tokens=False, truncation=True, max_length=PRE_MAX_LEN - 2,
              return_offsets_mapping=True)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]
    words = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    words = [w for w in words if w[0] < (offsets[-1][1] if offsets else 0)]
    hide, budget = set(), int(MASK_RATIO * len(words))
    numeric = [i for i, (a, b) in enumerate(words) if DIGIT.search(text[a:b])]
    while len(hide) < budget and words:
        pool = numeric if numeric and rng.random() < NUMERIC_START else range(len(words))
        start = rng.choice(list(pool))
        hide.update(range(start, min(start + rng.randint(SPAN_MIN, SPAN_MAX), len(words))))
    spans = [words[i] for i in hide]
    flags = [any(a < e and b > s for s, e in spans) for a, b in offsets]
    return [CLS] + ids + [SEP], [False] + flags + [False]


def pre_batch(texts, rng):
    rows = [masked_state(t, rng) for t in texts]
    width = max(len(r[0]) for r in rows)
    ids = [r[0] + [PAD] * (width - len(r[0])) for r in rows]
    att = [[1] * len(r[0]) + [0] * (width - len(r[0])) for r in rows]
    hid = [r[1] + [False] * (width - len(r[1])) for r in rows]
    as_t = lambda x, dt=None: torch.tensor(x, device=DEVICE, dtype=dt)
    return as_t(ids), as_t(att), as_t(hid, torch.bool)

## The JEPA model with LoRA

ModernBERT's weights are frozen; LoRA adapters (rank `LORA_RANK`) on its attention and MLP projections are the only trained encoder weights. The target is a copy whose adapters follow the context encoder's by EMA (the frozen base is the same on both sides). The retrieval check compares each prediction with up to 256 hidden tokens' targets in the batch: top-1 means its own target scored highest.

In [ ]:
from peft import LoraConfig, get_peft_model


class LoraJEPA(nn.Module):
    def __init__(self):
        super().__init__()
        base = AutoModel.from_pretrained(ENCODER, attn_implementation="sdpa")
        config = LoraConfig(r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0.0,
                            target_modules=["Wqkv", "Wo", "Wi"])
        self.encoder = get_peft_model(base, config)
        hidden = base.config.hidden_size
        layer = nn.TransformerEncoderLayer(hidden, 12, 2 * hidden, batch_first=True,
                                           norm_first=True)
        self.predictor = nn.TransformerEncoder(layer, num_layers=2)
        self.marker = nn.Parameter(torch.zeros(hidden))
        self.target = copy.deepcopy(self.encoder).requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        self.target.eval()
        return self

    def loss(self, ids, att, hid):
        h = self.encoder(input_ids=ids.masked_fill(hid, MASK), attention_mask=att).last_hidden_state
        pred = self.predictor(h + hid.unsqueeze(-1) * self.marker, src_key_padding_mask=att == 0)
        with torch.no_grad():
            t = self.target(input_ids=ids, attention_mask=att).last_hidden_state
            t = F.layer_norm(t.float(), t.shape[-1:])
        if not hid.any():
            return pred.sum() * 0, {}
        p, tt = pred[hid].float(), t[hid]
        cos = F.cosine_similarity(p, tt, dim=-1)
        with torch.no_grad():
            pick = torch.randperm(len(p), device=p.device)[:256]
            sims = F.normalize(p[pick], dim=-1) @ F.normalize(tt[pick], dim=-1).T
            top1 = (sims.argmax(1) == torch.arange(len(pick), device=p.device)).float().mean()
        return 1 - cos.mean(), {"cos": cos.mean().item(), "target std": tt.std(0).mean().item(),
                                "retrieval top-1": top1.item(), "chance": 1 / len(pick)}

## Stage 0 training

One schedule over `PRE_EPOCHS` passes of the states; after every step the target's adapters move towards the encoder's (EMA). Every 100 steps: the loss averaged over the last 100 batches, the cosine, the target's spread and the retrieval check.

In [ ]:
def pretrain():
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = LoraJEPA().to(DEVICE)
    model.train()
    steps = PRE_EPOCHS * math.ceil(len(pre_states) / PRE_BATCH)
    lora = [p for p in model.encoder.parameters() if p.requires_grad]
    rest = list(model.predictor.parameters()) + [model.marker]
    opt = torch.optim.AdamW(lora + rest, lr=LR_HEAD, weight_decay=0.01)
    warm = max(1, int(WARMUP * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    pairs = [(pt, ps) for (n, pt), ps in zip(model.target.named_parameters(),
                                             model.encoder.parameters()) if "lora_" in n]
    rng, recent, step, t0 = random.Random(SEED), [], 0, time.perf_counter()
    for _ in range(PRE_EPOCHS):
        for b in range(0, len(pre_states), PRE_BATCH):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
                loss, info = model.loss(*pre_batch(pre_states[b:b + PRE_BATCH], rng))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
            opt.step()
            sched.step()
            step += 1
            m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / steps) + 1) / 2
            with torch.no_grad():
                for pt, ps in pairs:
                    pt.mul_(m).add_(ps.detach(), alpha=1 - m)
            recent = (recent + [loss.item()])[-100:]
            if step % 100 == 0 or step == steps:
                extra = " ".join(f"{k} {v:.3f}" for k, v in info.items())
                log(f"JEPA-LoRA step {step}/{steps} | loss, last 100 "
                    f"{statistics.mean(recent):.4f} | {extra}")
    log(f"JEPA-LoRA pre-training done in {(time.perf_counter() - t0) / 60:.1f} min")
    return model

## Merge the adapters at two strengths

The adapters' change to each weight, added to plain ModernBERT in full (1.0) and at half strength (0.5); then the JEPA model is freed.

In [ ]:
jepa = pretrain()
plain = {k: v.detach().cpu().clone()
         for k, v in AutoModel.from_pretrained(ENCODER).state_dict().items()}
merged = {k: v.detach().cpu().clone()
          for k, v in jepa.encoder.merge_and_unload().state_dict().items()}
encoders = {f"LoRA-JEPA {s:g}": {k: plain[k] + s * (merged[k] - plain[k]) for k in plain}
            for s in MERGE_SCALES}
changed = sum((merged[k] - plain[k]).abs().sum().item() > 0 for k in plain)
log(f"merged: {changed} of {len(plain)} weight tensors changed by the adapters")
del jepa, merged
gc.collect()
torch.cuda.empty_cache()

## Typed Decisions' train split (never trained on)

Each arm is scored on it at the end. For reference: our 4B decoder 0.720, plain ModernBERT (run 8 B) 0.350, chance 0.318.

In [ ]:
td_path = hf_hub_download("LocalLLaMA/typed-decisions", "all/train-00000-of-00001.parquet",
                          repo_type="dataset")
td_items = []
for row in pd.read_parquet(td_path).to_dict("records"):
    state, qs, gold = (json.loads(row[k]) for k in ("state", "questions", "gold"))
    for qid, q in qs.items():
        names, _ = option_texts(q)
        probs = gold[qid]["probabilities"]
        if len(names) <= MAX_OPTIONS and all(n in probs for n in names):
            td_items.append(make_item(q["type"], q.get("instructions"), state, q,
                                      [float(probs[n]) for n in names], row["workflow"],
                                      f"{row['workflow']}/{qid}"))
print(f"Typed Decisions train: {len(td_items)} questions")

## Stage 1 from each merged encoder

The same decision training as run 8's arm B, from ModernBERT with the JEPA adapters merged at each strength.

In [ ]:
mixed = ts_items + train_items
arms = list(encoders.items())
curves, td_scores = {}, {}
for name, encoder in arms:
    model, curves[name] = run(name, mixed, encoder_state=encoder)
    td_scores[name] = evaluate(model, td_items, TRAIN_MAX_LEN, f"{name} on Typed Decisions train")
    del model
    gc.collect()
    torch.cuda.empty_cache()

## The comparison

Each arm's learning curve, then the final scores next to plain ModernBERT (run 8 B) and full-update JEPA (run 9), all with the same Stage 1.

In [ ]:
def chance(items):
    return round(statistics.mean(1 / len(it["names"]) for it in items), 3)


for name, curve in curves.items():
    print(f"{name:14s}{'point':>11s} {'train':>7s} {'dev':>7s} {'JevBench':>9s} "
          f"{'original':>9s} {'easy':>7s} {'hard':>7s}")
    for where, tr, dv, jv in curve:
        splits = " ".join(f"{jv['source'][s]['acc']:>7.3f}" for s in ("original", "easy", "hard"))
        print(f"{'':14s}{where:>11s} {tr['all']['acc']:>7.3f} {dv['all']['acc']:>7.3f} "
              f"{jv['all']['acc']:>9.3f}  {splits}")
print(f"\n{'final':14s} {'JevBench':>9s} {'easy':>6s} {'dev':>6s} {'TD train':>9s}")
print(f"{'plain':14s} {0.394:>9.3f} {0.562:>6.3f} {0.457:>6.3f} {0.350:>9.3f}   (run 8 B)")
print(f"{'JEPA, full':14s} {0.307:>9.3f} {0.271:>6.3f} {0.405:>6.3f} {0.347:>9.3f}   (run 9)")
for name, curve in curves.items():
    jv, dv = curve[-1][3], curve[-1][2]
    print(f"{name:14s} {jv['all']['acc']:>9.3f} {jv['source']['easy']['acc']:>6.3f} "
          f"{dv['all']['acc']:>6.3f} {td_scores[name]['all']['acc']:>9.3f}")
print(f"chance: JevBench {chance(jev_items)} | dev {chance(dev_items)} | TD {chance(td_items)}")